# Phase 2 — Activity 1: Data Preparation

**Project:** Explainable Disease Progression · Cyprus PROTEAS Dataset  
**Objective:** Build data inventory, create stratified patient-level splits, validate everything  
**Duration:** ~5 minutes locally (no GPU needed)  
**Predecessor:** Phase 1 (COMPLETE)

---

## What This Notebook Does

1. **Scans** all 45 patient directories → builds a complete inventory of 186 MRI scans
2. **Identifies** which scans have expert tumor masks (170 out of 186)
3. **Creates** stratified 3-fold and 5-fold patient-level cross-validation splits
4. **Validates** split integrity (no data leakage, a/b patients grouped correctly)
5. **Saves** everything as portable JSON (works both locally AND on Kaggle)

### Why This Matters

The `data_splits.json` produced here is the **contract** between Phase 2 (CNN) and Phase 3 (ViT).  
Both models MUST use the exact same train/test patients to make the comparison fair.

### Key Design Decisions

- **Patient-level splits** (not scan-level): prevents data leakage from the same brain
- **a/b patients together**: P04a and P04b are the same person (different lesions) → same fold
- **Stratified by treatment × histology**: ensures each fold has RS and FSRT patients
- **Relative paths**: all file paths in JSON are relative to DATA_ROOT → portable across machines

---
## 0. Setup

In [1]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# Auto-detect environment
KAGGLE_PATHS = [
    Path("/kaggle/input/cyprus-proteas-brain-mets"),
    Path("/kaggle/input/cyprus-proteas"),
]
LOCAL_PATH = Path("/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Data/Cyprus-PROTEAS-zips")

DATA_ROOT = None
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_ROOT = p
        ENV = 'kaggle'
        break
if DATA_ROOT is None:
    DATA_ROOT = LOCAL_PATH
    ENV = 'local'

OUTPUT_ROOT = Path("/kaggle/working") if ENV == 'kaggle' else Path("/home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Environment: {ENV}")
print(f"Data root:   {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Environment: local
Data root:   /home/moamed/canada_me/explainable_diseas/implementation_cyprus/Data/Cyprus-PROTEAS-zips
Output root: /home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs


---
## 1. Build Data Inventory

We scan every patient directory to find:
- All BraTS MRI modalities (T1, T1c, T2, FLAIR)
- All tumor segmentation masks
- Which timepoints are missing masks

**Important quirks in the data:**
- P01 and P30 use `"tumor segmentation"` (with space), others use `"tumor_segmentation"` (underscore)
- P31 has BraTS directories named `Fu1`, `Fu2`, `Fu3` (capital F) but mask files use lowercase `fu1`
- 5 patients have a/b splits: P04, P07, P17, P20, P23 (same person, different lesions)

In [2]:
def build_inventory(data_root: Path) -> pd.DataFrame:
    """
    Scan the dataset and build a complete inventory of all MRI scans and masks.
    Handles: tumor dir naming inconsistency, case-insensitive mask matching.
    """
    records = []
    
    for pdir in sorted(data_root.iterdir()):
        if not pdir.is_dir() or not pdir.name.startswith('P'):
            continue
        
        patient_dir = pdir.name
        
        # Group a/b patients: P04a, P04b → group P04
        match = re.match(r'(P\d+)([ab])?', patient_dir)
        patient_group = match.group(1) if match else patient_dir
        
        # Find BraTS directory
        brats_dir = pdir / 'BraTS'
        if not brats_dir.exists():
            print(f"  [WARN] {patient_dir}: No BraTS directory, skipping")
            continue
        
        # Find tumor mask directory (handles space vs underscore)
        tumor_dir = None
        for name in ['tumor_segmentation', 'tumor segmentation']:
            candidate = pdir / name
            if candidate.exists():
                tumor_dir = candidate
                break
        
        # Iterate over visits (baseline, fu1, fu2, ...)
        for visit_dir in sorted(brats_dir.iterdir()):
            if not visit_dir.is_dir():
                continue
            
            visit = visit_dir.name  # 'baseline', 'fu1', 'Fu1', etc.
            
            # Find modalities
            modalities = {}
            for mod_file in sorted(visit_dir.iterdir()):
                if mod_file.name.endswith('.nii.gz'):
                    mod_name = mod_file.stem.replace('.nii', '')
                    modalities[mod_name] = str(mod_file.relative_to(data_root))
            
            # Find mask (case-insensitive: P31 has Fu1 but mask uses fu1)
            mask_path = None
            has_mask = False
            if tumor_dir is not None:
                for visit_variant in [visit, visit.lower()]:
                    mask_name = f"{patient_dir}_tumor_mask_{visit_variant}.nii.gz"
                    mask_file = tumor_dir / mask_name
                    if mask_file.exists():
                        mask_path = str(mask_file.relative_to(data_root))
                        has_mask = True
                        break
            
            records.append({
                'patient_dir': patient_dir,
                'patient_group': patient_group,
                'visit': visit,
                'n_modalities': len(modalities),
                'has_mask': has_mask,
                't1_path': modalities.get('t1', None),
                't1c_path': modalities.get('t1c', None),
                't2_path': modalities.get('t2', None),
                'fla_path': modalities.get('fla', None),
                'mask_path': mask_path,
            })
    
    return pd.DataFrame(records)


# Build it
inventory = build_inventory(DATA_ROOT)
print(f"Total scan entries:       {len(inventory)}")
print(f"With expert masks:        {inventory['has_mask'].sum()}")
print(f"Without masks:            {(~inventory['has_mask']).sum()}")
print(f"Patient directories:      {inventory['patient_dir'].nunique()}")
print(f"Unique patient groups:    {inventory['patient_group'].nunique()}")
print(f"Modalities per scan:      {inventory['n_modalities'].unique().tolist()}")

Total scan entries:       186
With expert masks:        170
Without masks:            16
Patient directories:      45
Unique patient groups:    40
Modalities per scan:      [4]


### 1.1 Inspect Missing Masks

These timepoints have BraTS MRI data but no expert tumor annotations.  
They are excluded from segmentation training but will be used for embedding extraction later.

In [3]:
no_mask = inventory[inventory['has_mask'] == False][['patient_dir', 'visit']]
print(f"Timepoints WITHOUT expert masks ({len(no_mask)}):")
print()
for _, row in no_mask.iterrows():
    print(f"  {row['patient_dir']:6s} / {row['visit']}")

Timepoints WITHOUT expert masks (16):

  P09    / fu2
  P15    / fu1
  P15    / fu4
  P16    / fu1
  P16    / fu2
  P16    / fu3
  P19    / fu4
  P19    / fu5
  P20b   / fu1
  P20b   / fu2
  P29    / fu5
  P32    / fu2
  P32    / fu3
  P32    / fu4
  P32    / fu5
  P38    / fu3


### 1.2 Verify a/b Patient Groups

5 patients have two treated lesions (a/b directories). They MUST stay in the same fold.

In [4]:
# Find a/b groups
ab_groups = {}
for pid in sorted(inventory['patient_dir'].unique()):
    match = re.match(r'(P\d+)([ab])', pid)
    if match:
        base = match.group(1)
        ab_groups.setdefault(base, []).append(pid)

print("Patients with a/b splits (same person, different lesions):")
for base, dirs in sorted(ab_groups.items()):
    n_scans = inventory[inventory['patient_dir'].isin(dirs)]['has_mask'].sum()
    print(f"  {base}: {dirs} → {n_scans} scans with masks")

AB_GROUPS = list(ab_groups.keys())
print(f"\nThese {len(AB_GROUPS)} groups will always be kept in the same fold.")

Patients with a/b splits (same person, different lesions):
  P04: ['P04a', 'P04b'] → 10 scans with masks
  P07: ['P07a', 'P07b'] → 6 scans with masks
  P17: ['P17a', 'P17b'] → 11 scans with masks
  P20: ['P20a', 'P20b'] → 8 scans with masks
  P23: ['P23a', 'P23b'] → 9 scans with masks

These 5 groups will always be kept in the same fold.


### 1.3 Save Inventory

In [5]:
inv_path = OUTPUT_ROOT / 'data_inventory.csv'
inventory.to_csv(inv_path, index=False)
print(f"Saved: {inv_path} ({inv_path.stat().st_size / 1024:.1f} KB)")
inventory.head(10)

Saved: /home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs/data_inventory.csv (31.1 KB)


,patient_dir,patient_group,visit,n_modalities,has_mask,t1_path,t1c_path,t2_path,fla_path,mask_path
0,P01,P01,baseline,4,True,P01/BraTS/baseline/t1.nii.gz,P01/BraTS/baseline/t1c.nii.gz,P01/BraTS/baseline/t2.nii.gz,P01/BraTS/baseline/fla.nii.gz,P01/tumor segmentation/P01_tumor_mask_baseline...
1,P01,P01,fu1,4,True,P01/BraTS/fu1/t1.nii.gz,P01/BraTS/fu1/t1c.nii.gz,P01/BraTS/fu1/t2.nii.gz,P01/BraTS/fu1/fla.nii.gz,P01/tumor segmentation/P01_tumor_mask_fu1.nii.gz
2,P01,P01,fu2,4,True,P01/BraTS/fu2/t1.nii.gz,P01/BraTS/fu2/t1c.nii.gz,P01/BraTS/fu2/t2.nii.gz,P01/BraTS/fu2/fla.nii.gz,P01/tumor segmentation/P01_tumor_mask_fu2.nii.gz
3,P01,P01,fu3,4,True,P01/BraTS/fu3/t1.nii.gz,P01/BraTS/fu3/t1c.nii.gz,P01/BraTS/fu3/t2.nii.gz,P01/BraTS/fu3/fla.nii.gz,P01/tumor segmentation/P01_tumor_mask_fu3.nii.gz
4,P01,P01,fu4,4,True,P01/BraTS/fu4/t1.nii.gz,P01/BraTS/fu4/t1c.nii.gz,P01/BraTS/fu4/t2.nii.gz,P01/BraTS/fu4/fla.nii.gz,P01/tumor segmentation/P01_tumor_mask_fu4.nii.gz
5,P02,P02,baseline,4,True,P02/BraTS/baseline/t1.nii.gz,P02/BraTS/baseline/t1c.nii.gz,P02/BraTS/baseline/t2.nii.gz,P02/BraTS/baseline/fla.nii.gz,P02/tumor_segmentation/P02_tumor_mask_baseline...
6,P02,P02,fu1,4,True,P02/BraTS/fu1/t1.nii.gz,P02/BraTS/fu1/t1c.nii.gz,P02/BraTS/fu1/t2.nii.gz,P02/BraTS/fu1/fla.nii.gz,P02/tumor_segmentation/P02_tumor_mask_fu1.nii.gz
7,P02,P02,fu2,4,True,P02/BraTS/fu2/t1.nii.gz,P02/BraTS/fu2/t1c.nii.gz,P02/BraTS/fu2/t2.nii.gz,P02/BraTS/fu2/fla.nii.gz,P02/tumor_segmentation/P02_tumor_mask_fu2.nii.gz
8,P03,P03,baseline,4,True,P03/BraTS/baseline/t1.nii.gz,P03/BraTS/baseline/t1c.nii.gz,P03/BraTS/baseline/t2.nii.gz,P03/BraTS/baseline/fla.nii.gz,P03/tumor_segmentation/P03_tumor_mask_baseline...
9,P03,P03,fu1,4,True,P03/BraTS/fu1/t1.nii.gz,P03/BraTS/fu1/t1c.nii.gz,P03/BraTS/fu1/t2.nii.gz,P03/BraTS/fu1/fla.nii.gz,P03/tumor_segmentation/P03_tumor_mask_fu1.nii.gz


---
## 2. Load Clinical Data for Stratification

We stratify by **treatment type × histology** to ensure each fold has a representative mix:
- Treatment: RS (36 patients) vs FSRT (11 patients)
- Histology: NSCLC (29) vs Breast (17) vs SCLC (1)

The single SCLC patient is merged with NSCLC for stratification (too few for its own stratum).

In [6]:
# Load clinical data
clinical_path = OUTPUT_ROOT / 'PROTEAS_Clinical_Cleaned.xlsx'
if not clinical_path.exists():
    clinical_path = OUTPUT_ROOT.parent / 'outputs' / 'PROTEAS_Clinical_Cleaned.xlsx'

clinical = pd.read_excel(clinical_path)

# Build stratification labels per patient GROUP
strat_labels = {}
for _, row in clinical.iterrows():
    pid = row['Patient ID (Zenodo)']
    treatment = row.get('Treatment_Group', 'Unknown')
    histology = row.get('Tumour Histology', 'Unknown')
    
    # Get patient group (P04a → P04)
    match = re.match(r'(P\d+)([ab])?', pid)
    group = match.group(1) if match else pid
    
    hist_simple = 'NSCLC' if 'NSCLC' in str(histology) else (
        'Breast' if 'Breast' in str(histology) else 'Other'
    )
    label = f"{treatment}_{hist_simple}"
    
    if group not in strat_labels:
        strat_labels[group] = label

# Display distribution
label_counts = Counter(strat_labels.values())
print("Stratification distribution:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:20s} {count:3d} patients ({count/len(strat_labels)*100:.1f}%)")

Stratification distribution:
  RS_NSCLC              25 patients (62.5%)
  FSRT_Breast            8 patients (20.0%)
  RS_Breast              6 patients (15.0%)
  RS_Other               1 patients (2.5%)


---
## 3. Create Stratified Patient-Level Splits

We create **both** 3-fold and 5-fold splits now. Our progressive strategy:

```
Day 1:   Single split (fold 0) → quick test, ~2 hrs
Week 1:  3-fold CV → working results with mean ± std, ~16 hrs
Later:   5-fold CV → publication-quality numbers, ~27 hrs
```

**Rules:**
1. Split at patient GROUP level (P04a + P04b always together)
2. Stratify by treatment × histology
3. Only include timepoints that have masks
4. Same seed (42) so splits are reproducible

In [7]:
from sklearn.model_selection import StratifiedKFold

SEED = 42

# Get patient groups with at least one mask
masked = inventory[inventory['has_mask'] == True]
groups_with_masks = sorted(masked['patient_group'].unique())
print(f"Patient groups with masks: {len(groups_with_masks)}")

# Build labels (merge SCLC/Other into NSCLC for stratification)
labels = []
for g in groups_with_masks:
    label = strat_labels.get(g, 'Unknown')
    if 'Other' in label or 'SCLC' in label:
        prefix = label.split('_')[0]
        label = f"{prefix}_NSCLC"
    labels.append(label)

groups_array = np.array(groups_with_masks)
labels_array = np.array(labels)


def expand_groups(groups, inventory_df):
    """Expand patient groups to directory names: P04 → [P04a, P04b]"""
    dirs = []
    all_dirs = inventory_df['patient_dir'].unique()
    for g in groups:
        matching = [d for d in all_dirs if re.match(rf'^{g}[ab]?$', d)]
        dirs.extend(sorted(matching))
    return dirs


def get_scan_entries(patient_dirs, masked_df):
    """Get scan entries for given patient directories (only masked scans)."""
    scans = []
    for _, row in masked_df[masked_df['patient_dir'].isin(patient_dirs)].iterrows():
        scans.append({
            'patient_dir': row['patient_dir'],
            'patient_group': row['patient_group'],
            'visit': row['visit'],
            't1': row['t1_path'],
            't1c': row['t1c_path'],
            't2': row['t2_path'],
            'fla': row['fla_path'],
            'mask': row['mask_path'],
        })
    return scans

Patient groups with masks: 40


### 3.1 Create 3-Fold Splits (Primary)

In [8]:
skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
fold_3 = {}

print("3-Fold Cross-Validation Splits:")
print("=" * 70)

for i, (train_idx, test_idx) in enumerate(skf3.split(groups_array, labels_array)):
    train_groups = groups_array[train_idx].tolist()
    test_groups = groups_array[test_idx].tolist()
    
    train_dirs = expand_groups(train_groups, inventory)
    test_dirs = expand_groups(test_groups, inventory)
    
    train_scans = get_scan_entries(train_dirs, masked)
    test_scans = get_scan_entries(test_dirs, masked)
    
    fold_3[f'fold_{i}'] = {
        'train_groups': train_groups,
        'test_groups': test_groups,
        'train_dirs': train_dirs,
        'test_dirs': test_dirs,
        'train_scans': train_scans,
        'test_scans': test_scans,
        'n_train_scans': len(train_scans),
        'n_test_scans': len(test_scans),
    }
    
    # Show details
    train_labels = [strat_labels.get(g, '?') for g in train_groups]
    test_labels = [strat_labels.get(g, '?') for g in test_groups]
    
    print(f"\nFold {i}:")
    print(f"  Train: {len(train_groups)} groups → {len(train_scans)} scans")
    print(f"         Strat: {dict(Counter(train_labels))}")
    print(f"  Test:  {len(test_groups)} groups → {len(test_scans)} scans")
    print(f"         Strat: {dict(Counter(test_labels))}")
    print(f"         Groups: {test_groups}")

3-Fold Cross-Validation Splits:

Fold 0:
  Train: 26 groups → 114 scans
         Strat: {'RS_NSCLC': 17, 'RS_Breast': 4, 'FSRT_Breast': 5}
  Test:  14 groups → 56 scans
         Strat: {'RS_NSCLC': 8, 'RS_Other': 1, 'FSRT_Breast': 3, 'RS_Breast': 2}
         Groups: ['P01', 'P03', 'P08', 'P12', 'P14', 'P17', 'P19', 'P21', 'P22', 'P31', 'P32', 'P34', 'P39', 'P40']

Fold 1:
  Train: 27 groups → 119 scans
         Strat: {'RS_NSCLC': 16, 'FSRT_Breast': 6, 'RS_Other': 1, 'RS_Breast': 4}
  Test:  13 groups → 51 scans
         Strat: {'RS_NSCLC': 9, 'RS_Breast': 2, 'FSRT_Breast': 2}
         Groups: ['P02', 'P05', 'P06', 'P07', 'P09', 'P13', 'P16', 'P18', 'P20', 'P24', 'P33', 'P35', 'P37']

Fold 2:
  Train: 27 groups → 107 scans
         Strat: {'RS_NSCLC': 17, 'RS_Breast': 4, 'FSRT_Breast': 5, 'RS_Other': 1}
  Test:  13 groups → 63 scans
         Strat: {'RS_NSCLC': 8, 'FSRT_Breast': 3, 'RS_Breast': 2}
         Groups: ['P04', 'P10', 'P11', 'P15', 'P23', 'P25', 'P26', 'P27', 'P28', 'P29', '

### 3.2 Create 5-Fold Splits (For Later / Publication)

In [9]:
skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_5 = {}

print("5-Fold Cross-Validation Splits:")
print("=" * 70)

for i, (train_idx, test_idx) in enumerate(skf5.split(groups_array, labels_array)):
    train_groups = groups_array[train_idx].tolist()
    test_groups = groups_array[test_idx].tolist()
    
    train_dirs = expand_groups(train_groups, inventory)
    test_dirs = expand_groups(test_groups, inventory)
    
    train_scans = get_scan_entries(train_dirs, masked)
    test_scans = get_scan_entries(test_dirs, masked)
    
    fold_5[f'fold_{i}'] = {
        'train_groups': train_groups,
        'test_groups': test_groups,
        'train_dirs': train_dirs,
        'test_dirs': test_dirs,
        'train_scans': train_scans,
        'test_scans': test_scans,
        'n_train_scans': len(train_scans),
        'n_test_scans': len(test_scans),
    }
    
    print(f"  Fold {i}: train={len(train_groups)} groups ({len(train_scans)} scans) | "
          f"test={len(test_groups)} groups ({len(test_scans)} scans)")

5-Fold Cross-Validation Splits:
  Fold 0: train=32 groups (143 scans) | test=8 groups (27 scans)
  Fold 1: train=32 groups (132 scans) | test=8 groups (38 scans)
  Fold 2: train=32 groups (138 scans) | test=8 groups (32 scans)
  Fold 3: train=32 groups (142 scans) | test=8 groups (28 scans)
  Fold 4: train=32 groups (125 scans) | test=8 groups (45 scans)


---
## 4. Validate Splits

Critical checks:
1. No patient appears in BOTH train and test in the same fold
2. a/b patients are always in the same fold
3. Every patient is tested exactly once across all folds
4. All file paths in the splits actually exist on disk

In [10]:
def validate_splits(splits_dict, cv_name):
    """Validate split integrity."""
    all_test_groups = []
    
    for fold_name, fold_data in splits_dict.items():
        train_g = set(fold_data['train_groups'])
        test_g = set(fold_data['test_groups'])
        
        # Check 1: No overlap
        overlap = train_g & test_g
        assert len(overlap) == 0, f"{fold_name}: Train/test overlap: {overlap}"
        
        # Check 2: a/b patients together
        for group in AB_GROUPS:
            in_train = group in train_g
            in_test = group in test_g
            assert not (in_train and in_test), f"{group} split across train/test in {fold_name}!"
        
        all_test_groups.extend(fold_data['test_groups'])
    
    # Check 3: Every patient tested exactly once
    test_counts = Counter(all_test_groups)
    for g, c in test_counts.items():
        assert c == 1, f"{cv_name}: {g} appears {c} times in test folds"
    
    # Check 4: All paths resolve
    n_checked = 0
    n_errors = 0
    for fold_name, fold_data in splits_dict.items():
        for scan in fold_data['train_scans'] + fold_data['test_scans']:
            for key in ['t1', 't1c', 't2', 'fla', 'mask']:
                full_path = DATA_ROOT / scan[key]
                n_checked += 1
                if not full_path.exists():
                    print(f"  ❌ Missing: {scan['patient_dir']}/{scan['visit']}/{key}")
                    n_errors += 1
    
    print(f"{cv_name}: ✅ No train/test overlap")
    print(f"{cv_name}: ✅ a/b groups intact")
    print(f"{cv_name}: ✅ All {len(test_counts)} patients tested exactly once")
    print(f"{cv_name}: ✅ {n_checked} paths checked, {n_errors} errors")


validate_splits(fold_3, '3-fold')
print()
validate_splits(fold_5, '5-fold')
print("\n✅ ALL VALIDATIONS PASSED")

3-fold: ✅ No train/test overlap
3-fold: ✅ a/b groups intact
3-fold: ✅ All 40 patients tested exactly once
3-fold: ✅ 2550 paths checked, 0 errors

5-fold: ✅ No train/test overlap
5-fold: ✅ a/b groups intact
5-fold: ✅ All 40 patients tested exactly once
5-fold: ✅ 4250 paths checked, 0 errors

✅ ALL VALIDATIONS PASSED


---
## 5. Save Splits to JSON

This is the **core output** of Activity 1.  
Upload `data_splits.json` to Kaggle alongside the NIfTI data.

In [11]:
# Build final splits object
splits = {
    '3fold': fold_3,
    '5fold': fold_5,
    'metadata': {
        'total_patient_groups': len(groups_with_masks),
        'total_scans_with_masks': int(masked.shape[0]),
        'total_scans_all': int(len(inventory)),
        'stratification_distribution': dict(Counter(strat_labels.values())),
        'ab_patient_groups': AB_GROUPS,
        'seed': SEED,
        'note': 'All paths are RELATIVE to DATA_ROOT. Resolve at runtime.',
    }
}

# Save
splits_path = OUTPUT_ROOT / 'data_splits.json'
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Saved: {splits_path}")
print(f"Size:  {splits_path.stat().st_size / 1024:.1f} KB")

# Also save summary
summary = {
    'total_patient_dirs': int(inventory['patient_dir'].nunique()),
    'total_patient_groups': int(inventory['patient_group'].nunique()),
    'total_scans': int(len(inventory)),
    'scans_with_masks': int(inventory['has_mask'].sum()),
    'scans_without_masks': int((~inventory['has_mask']).sum()),
    'ab_groups': AB_GROUPS,
    'modalities': ['t1', 't1c', 't2', 'fla'],
    'visits': sorted(inventory['visit'].unique().tolist()),
}
summary_path = OUTPUT_ROOT / 'phase2_data_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Saved: {summary_path}")

Saved: /home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs/data_splits.json
Size:  500.9 KB
Saved: /home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs/phase2_data_summary.json


---
## 6. Visual Summary

Quick visual check of the split distributions.

In [12]:
# Summary table
print("\n" + "=" * 60)
print("  ACTIVITY 1 — DATA PREPARATION COMPLETE")
print("=" * 60)

print(f"\n  Dataset: Cyprus PROTEAS Brain Metastases")
print(f"  Total scans:           {len(inventory)}")
print(f"  With expert masks:     {inventory['has_mask'].sum()} (used for training)")
print(f"  Without masks:         {(~inventory['has_mask']).sum()} (excluded)")
print(f"  Patient groups:        {inventory['patient_group'].nunique()}")
print(f"  a/b paired patients:   {len(AB_GROUPS)} ({AB_GROUPS})")

print(f"\n  3-Fold Split:")
for k, v in fold_3.items():
    print(f"    {k}: train={v['n_train_scans']:3d} scans | test={v['n_test_scans']:2d} scans")

print(f"\n  5-Fold Split:")
for k, v in fold_5.items():
    print(f"    {k}: train={v['n_train_scans']:3d} scans | test={v['n_test_scans']:2d} scans")

print(f"\n  Output files:")
print(f"    data_splits.json      ({splits_path.stat().st_size/1024:.1f} KB) ← UPLOAD TO KAGGLE")
print(f"    data_inventory.csv    ({inv_path.stat().st_size/1024:.1f} KB)")
print(f"    phase2_data_summary.json")

print(f"\n  Next step: Run Activity 2 notebook (Quick Test on Kaggle)")


  ACTIVITY 1 — DATA PREPARATION COMPLETE

  Dataset: Cyprus PROTEAS Brain Metastases
  Total scans:           186
  With expert masks:     170 (used for training)
  Without masks:         16 (excluded)
  Patient groups:        40
  a/b paired patients:   5 (['P04', 'P07', 'P17', 'P20', 'P23'])

  3-Fold Split:
    fold_0: train=114 scans | test=56 scans
    fold_1: train=119 scans | test=51 scans
    fold_2: train=107 scans | test=63 scans

  5-Fold Split:
    fold_0: train=143 scans | test=27 scans
    fold_1: train=132 scans | test=38 scans
    fold_2: train=138 scans | test=32 scans
    fold_3: train=142 scans | test=28 scans
    fold_4: train=125 scans | test=45 scans

  Output files:
    data_splits.json      (500.9 KB) ← UPLOAD TO KAGGLE
    data_inventory.csv    (31.1 KB)
    phase2_data_summary.json

  Next step: Run Activity 2 notebook (Quick Test on Kaggle)
